# Truy cập file

# Đọc dữ liệu và concat 2 sheet

In [1]:
import os

# Polars
os.environ["POLARS_MAX_THREADS"] = "1"

# PyArrow
os.environ["ARROW_NUM_THREADS"] = "1"

# BLAS / OpenMP (NumPy / Pandas / SciPy / sklearn backend)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [2]:
import polars as pl
import pandas as pd

df1 = pd.read_excel("./input/20250715-ketquathi-ct2018a.xlsx", sheet_name="Sheet1")
df1.head()

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1,1000001,5.75,7.75,NaN,7.75,8.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1000002,8.00,8.25,8.50,6.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1000003,6.75,8.50,8.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1000004,5.25,7.50,6.50,5.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1000005,NaN,7.00,NaN,NaN,NaN,NaN,NaN,NaN,5.5,6.25,NaN,NaN,NaN


In [3]:
df2 = pd.read_excel("./input/20250715-ketquathi-ct2018a.xlsx", sheet_name="Sheet2")
df2.head()

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1000001,52013349,3.25,6.75,5.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.5,N1
1,1000002,52013350,3.25,5.25,6.00,3.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1000003,52013351,4.50,7.00,4.10,3.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1000004,52013352,4.25,5.75,7.00,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1000005,52013353,4.25,8.00,4.35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.5,N1


In [4]:
df_score = pd.concat([df1, df2])
df_score.head()

,STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
0,1,1000001,5.75,7.75,NaN,7.75,8.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1000002,8.00,8.25,8.50,6.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1000003,6.75,8.50,8.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1000004,5.25,7.50,6.50,5.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1000005,NaN,7.00,NaN,NaN,NaN,NaN,NaN,NaN,5.5,6.25,NaN,NaN,NaN


In [5]:
import gc

del df1, df2
gc.collect()

90

In [6]:
df_score.to_parquet(
    "./output/20250715-ketquathi-ct2018a.parquet",
    compression="zstd",
    compression_level=6,
    index=False
)

In [7]:
df_score = pl.scan_parquet("./output/20250715-ketquathi-ct2018a.parquet")
if "__index_level_0__" in df_score.columns:
    df_score = df_score.drop("__index_level_0__")
df_score.head().collect()

C:\Users\musba\AppData\Local\Temp\ipykernel_7016\741271801.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if "__index_level_0__" in df_score.columns:


STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin học,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
1,1000001,5.75,7.75,null,7.75,8.25,null,null,null,null,null,null,null,null
2,1000002,8.0,8.25,8.5,6.75,null,null,null,null,null,null,null,null,null
3,1000003,6.75,8.5,8.75,null,null,null,null,null,null,null,null,null,null
4,1000004,5.25,7.5,6.5,5.5,null,null,null,null,null,null,null,null,null
5,1000005,null,7.0,null,null,null,null,null,null,5.5,6.25,null,null,null


In [8]:
df_combinations_table = pl.read_csv("./input/bang_to_hop_mon.csv")
df_combinations_table.head()

Môn 1,Môn 2,Môn 3,Tổ hợp
str,str,str,str
"""Toán""","""Lí""","""Hóa""","""A00"""
"""Toán""","""Lí""","""Anh""","""A01"""
"""Toán""","""Lí""","""Sinh""","""A02"""
"""Toán""","""Lí""","""Sử""","""A03"""
"""Toán""","""Lí""","""Địa""","""A04"""


In [9]:
!pip install xlsxwriter

In [10]:
df_score.head(10).collect().write_excel("./output/head.xlsx")

# Chuẩn hóa

## Step 1: Xử lý điểm môn Toán, Văn - Chuẩn hóa SOBAODANH

In [11]:
complusory_subject = ["Toán", "Văn"]
optional_subject = [i for i in df_score.collect_schema().names() if i not in ["STT", "SOBAODANH", "Toán", "Văn", "Mã môn ngoại ngữ", ""]]

optional_subject

['Lí',
 'Hóa',
 'Sinh',
 'Tin học',
 'Công nghệ công nghiệp',
 'Công nghệ nông nghiệp',
 'Sử',
 'Địa',
 'Giáo dục kinh tế và pháp luật',
 'Ngoại ngữ']

In [12]:
# Danh sách môn tự chọn (đã loại Toán, Văn)
all_subjects = ["Toán", "Văn"] + optional_subject

df_score_step_1 = df_score.with_columns([
    # 1. Xử lý SBD
    pl.col("SOBAODANH").cast(pl.String).str.zfill(8),

    # 2. Ép kiểu Float32 cho các môn Tự chọn (giữ nguyên null)
    *[pl.col(i).cast(pl.Float32) for i in optional_subject],

    # 3. Vừa ép kiểu, vừa fill_null cho Toán và Văn trong 1 nốt nhạc
    pl.col("Toán").cast(pl.Float32).fill_null(0),
    pl.col("Văn").cast(pl.Float32).fill_null(0),
])

# Chạy thử 5 dòng đầu
print(df_score_step_1.head().collect())

shape: (5, 15)
┌─────┬───────────┬──────┬──────┬───┬──────┬───────────────────────────────┬───────────┬──────────────────┐
│ STT ┆ SOBAODANH ┆ Toán ┆ Văn  ┆ … ┆ Địa  ┆ Giáo dục kinh tế và pháp luật ┆ Ngoại ngữ ┆ Mã môn ngoại ngữ │
│ --- ┆ ---       ┆ ---  ┆ ---  ┆   ┆ ---  ┆ ---                           ┆ ---       ┆ ---              │
│ i64 ┆ str       ┆ f32  ┆ f32  ┆   ┆ f32  ┆ f32                           ┆ f32       ┆ str              │
╞═════╪═══════════╪══════╪══════╪═══╪══════╪═══════════════════════════════╪═══════════╪══════════════════╡
│ 1   ┆ 01000001  ┆ 5.75 ┆ 7.75 ┆ … ┆ null ┆ null                          ┆ null      ┆ null             │
│ 2   ┆ 01000002  ┆ 8.0  ┆ 8.25 ┆ … ┆ null ┆ null                          ┆ null      ┆ null             │
│ 3   ┆ 01000003  ┆ 6.75 ┆ 8.5  ┆ … ┆ null ┆ null                          ┆ null      ┆ null             │
│ 4   ┆ 01000004  ┆ 5.25 ┆ 7.5  ┆ … ┆ null ┆ null                          ┆ null      ┆ null             │
│ 5   ┆ 01000

In [13]:
df_score_step_1 = df_score_step_1.rename(
    {"Tin học" : "Tin"}
)

print(df_score_step_1.collect_schema().names())

['STT', 'SOBAODANH', 'Toán', 'Văn', 'Lí', 'Hóa', 'Sinh', 'Tin', 'Công nghệ công nghiệp', 'Công nghệ nông nghiệp', 'Sử', 'Địa', 'Giáo dục kinh tế và pháp luật', 'Ngoại ngữ', 'Mã môn ngoại ngữ']


## Step 2: Loại bỏ người có điểm dưới 1

In [14]:
complusory_subject = ["Toán", "Văn"]
optional_subject = [i for i in df_score_step_1.collect_schema().names() if i not in ["STT", "SOBAODANH", "Toán", "Văn", "Mã môn ngoại ngữ", ""]]

optional_subject

['Lí',
 'Hóa',
 'Sinh',
 'Tin',
 'Công nghệ công nghiệp',
 'Công nghệ nông nghiệp',
 'Sử',
 'Địa',
 'Giáo dục kinh tế và pháp luật',
 'Ngoại ngữ']

In [15]:
# 1. Đếm số môn tự chọn thực tế dự thi (không null)
num_attempted = pl.sum_horizontal(
    pl.col(optional_subject).is_not_null().cast(pl.Int8)
)

# 2. Đếm số môn tự chọn đạt yêu cầu (không null và > 1.0)
num_passed = pl.sum_horizontal(
    (pl.col(optional_subject) > 1.0).cast(pl.Int8)
)

# 3. Logic Hợp lệ theo Quy chế mới (2+2):
# - Toán, Văn phải > 1.0
# - Số môn tự chọn ĐÃ THI phải đúng bằng 2
# - Cả 2 môn đó đều phải KHÔNG LIỆT
is_eligible_logic = (
    (pl.col("Toán") > 1.0) &
    (pl.col("Văn") > 1.0) &
    (num_attempted == 2) &
    (num_passed == 2)
)

df_score_step_2 = df_score_step_1.with_columns([
    # Cast điểm sang Float
    *[pl.col(i).cast(pl.Float64).alias(i) for i in optional_subject],

    # Tạo cột trạng thái: True là đủ điều kiện, False là rớt/không đủ bài
    is_eligible_logic.alias("is_eligible")
])

In [16]:
df_score_step_2.head().collect()

STT,SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Ngoại ngữ,Mã môn ngoại ngữ,is_eligible
i64,str,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,bool
1,"""01000001""",5.75,7.75,null,7.75,8.25,null,null,null,null,null,null,null,null,true
2,"""01000002""",8.0,8.25,8.5,6.75,null,null,null,null,null,null,null,null,null,true
3,"""01000003""",6.75,8.5,8.75,null,null,null,null,null,null,null,null,null,null,false
4,"""01000004""",5.25,7.5,6.5,5.5,null,null,null,null,null,null,null,null,null,true
5,"""01000005""",0.0,7.0,null,null,null,null,null,null,5.5,6.25,null,null,null,false


## Step 3: Tháo môn Ngoại ngữ thành n cột với n môn

In [17]:
ma_ngoai_ngu = {
    "N1": "Anh",
    "N2": "Nga",
    "N3": "Pháp",
    "N4": "Trung",
    "N5": "Đức",
    "N6": "Nhật",
    "N7": "Hàn"
}

In [18]:
ngoai_ngu = df_score_step_2 \
    .select(["SOBAODANH", "Ngoại ngữ", "Mã môn ngoại ngữ"]) \
    .filter(
        pl.col("Ngoại ngữ").is_not_null() &
        pl.col("Mã môn ngoại ngữ").is_not_null()
    )\
    .collect().pivot(
        on = "Mã môn ngoại ngữ",
        index = "SOBAODANH",
        values = "Ngoại ngữ"
    ).rename(ma_ngoai_ngu)

ngoai_ngu.head()

SOBAODANH,Anh,Nhật,Trung,Hàn,Đức,Pháp,Nga
str,f64,f64,f64,f64,f64,f64,f64
"""01000553""",7.5,null,null,null,null,null,null
"""01000555""",8.75,null,null,null,null,null,null
"""01000556""",7.0,null,null,null,null,null,null
"""01000557""",8.75,null,null,null,null,null,null
"""01000558""",7.5,null,null,null,null,null,null


In [19]:
df_score_step_3 = df_score_step_2 \
    .drop(["Ngoại ngữ", "Mã môn ngoại ngữ", "STT"]) \
    .join(ngoai_ngu.lazy(), on="SOBAODANH", how="left")

df_score_step_3.head().collect()

SOBAODANH,Toán,Văn,Lí,Hóa,Sinh,Tin,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,is_eligible,Anh,Nhật,Trung,Hàn,Đức,Pháp,Nga
str,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,f64,f64,f64,f64
"""01000001""",5.75,7.75,null,7.75,8.25,null,null,null,null,null,null,true,null,null,null,null,null,null,null
"""01000002""",8.0,8.25,8.5,6.75,null,null,null,null,null,null,null,true,null,null,null,null,null,null,null
"""01000003""",6.75,8.5,8.75,null,null,null,null,null,null,null,null,false,null,null,null,null,null,null,null
"""01000004""",5.25,7.5,6.5,5.5,null,null,null,null,null,null,null,true,null,null,null,null,null,null,null
"""01000005""",0.0,7.0,null,null,null,null,null,null,5.5,6.25,null,false,null,null,null,null,null,null,null


In [20]:
df_score_step_3.collect().write_parquet(
    "./output/bang_diem_da_xu_ly_2025.parquet",
    compression="zstd",
    compression_level=11,
    use_pyarrow=True
)

In [21]:
df_score_step_3.collect().write_csv("./output/bang_diem_da_xu_ly_2025.csv")

In [22]:
df_score_step_3.drop(["SOBAODANH", "is_eligible"]).describe()

statistic,Toán,Văn,Lí,Hóa,Sinh,Tin,Công nghệ công nghiệp,Công nghệ nông nghiệp,Sử,Địa,Giáo dục kinh tế và pháp luật,Anh,Nhật,Trung,Hàn,Đức,Pháp,Nga
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.131136e6,1.131136e6,347599.0,240135.0,69895.0,7602.0,2290.0,22048.0,481293.0,476472.0,246401.0,351848.0,491.0,4225.0,541.0,150.0,408.0,60.0
"""null_count""",0.0,0.0,783537.0,891001.0,1.061241e6,1.123534e6,1.128846e6,1.109088e6,649843.0,654664.0,884735.0,779288.0,1.130645e6,1.126911e6,1.130595e6,1.130986e6,1.130728e6,1.131076e6
"""mean""",4.762109,6.974806,6.98506,6.06488,5.778292,6.782761,5.792555,7.715897,6.519251,6.627846,7.691404,5.375019,6.289715,7.356391,5.895564,7.816667,8.376838,8.941667
"""std""",1.703135,1.348674,1.517309,1.810849,1.580777,1.482733,1.53987,1.170514,1.632873,1.749799,1.180889,1.449111,2.087915,2.033292,1.922382,1.641448,1.46792,1.156564
"""min""",0.0,0.0,0.0,0.75,0.85,1.35,2.3,1.55,0.0,0.0,1.25,0.0,2.0,1.5,1.25,3.5,3.25,5.25
"""25%""",3.5,6.25,5.85,4.75,4.6,5.75,4.55,7.0,5.25,5.35,7.0,4.25,4.5,5.75,4.25,6.75,7.75,8.75
"""50%""",4.6,7.25,7.0,6.0,5.75,6.75,5.6,7.75,6.6,6.75,7.75,5.25,6.5,7.75,5.75,8.25,8.75,9.25
"""75%""",5.85,8.0,8.25,7.5,7.0,7.85,7.0,8.5,7.75,8.0,8.5,6.25,8.0,9.25,7.25,9.0,9.5,9.75
"""max""",10.0,9.75,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0


## Step 4: Xác định các tổ hợp khả dĩ

In [23]:
subjects = [i for i in df_score_step_3.collect_schema() if i not in ["SOBAODANH", "is_eligible"]]

# B4.1. Xây dựng bảng các môn mà thí sinh đã thi
computation_subject = df_score_step_3.with_columns([
    pl.concat_list(
            [
                pl.when(pl.col(s).is_not_null())
                .then(pl.lit(s))
                .otherwise(None)
                for s in subjects
            ]
        )
    .list.drop_nulls()
    .alias("Danh sách môn thi")
])\
.select(["SOBAODANH", "Danh sách môn thi"])

computation_subject.head(10).collect()

SOBAODANH,Danh sách môn thi
str,list[str]
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]"
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]"
"""01000003""","[""Toán"", ""Văn"", ""Lí""]"
"""01000004""","[""Toán"", ""Văn"", … ""Hóa""]"
"""01000005""","[""Toán"", ""Văn"", … ""Địa""]"
"""01000006""","[""Toán"", ""Văn"", … ""Hóa""]"
"""01000007""","[""Toán"", ""Văn"", … ""Sinh""]"
"""01000008""","[""Toán"", ""Văn"", … ""Địa""]"
"""01000009""","[""Toán"", ""Văn"", … ""Hóa""]"


In [24]:
## B4.2. Tính toán 4 tổ hợp từ 3 trong 4 môn trong danh sách
import itertools

def get_combs(combs):
    if len(combs) < 3:
        return []
    # Ép kiểu về list của các list để Polars hiểu cấu trúc phân cấp
    return [list(c) for c in itertools.combinations(combs, 3)]


from collections import defaultdict

# Khởi tạo dict chứa List
mapping_dict = defaultdict(list)

# df_combinations_table là bảng chứa: Tổ hợp, Môn 1, Môn 2, Môn 3
for row in df_combinations_table.to_dicts():
    # Key là bộ 3 môn (không quan trọng thứ tự)
    key = frozenset([row["Môn 1"], row["Môn 2"], row["Môn 3"]])
    # Value là danh sách các mã tổ hợp (append thêm vào nếu trùng bộ môn)
    mapping_dict[key].append(row["Tổ hợp"])

In [25]:
# B4.2 & B4.3: Sinh và phân rã tổ hợp
df_combinations_exploded = computation_subject.with_columns(
    pl.col("Danh sách môn thi").map_elements(
        get_combs,
        return_dtype=pl.List(pl.List(pl.String))
    ).alias("Tổ hợp 3 môn")
).explode("Tổ hợp 3 môn").drop_nulls("Tổ hợp 3 môn")



# Tách list thành 3 cột môn học riêng biệt
df_final_step_1 = df_combinations_exploded.with_columns([
    pl.col("Tổ hợp 3 môn").list.get(0).alias("Môn 1"),
    pl.col("Tổ hợp 3 môn").list.get(1).alias("Môn 2"),
    pl.col("Tổ hợp 3 môn").list.get(2).alias("Môn 3")
])



# B4.5: Ánh xạ mã tổ hợp (A00, B00...) và lọc rác
df_final_step_2 = df_final_step_1.with_columns(
    pl.struct(["Môn 1", "Môn 2", "Môn 3"])
    .map_elements(
        # Tra cứu: nếu thấy thì trả về List mã, không thấy thì List rỗng
        lambda x: mapping_dict.get(frozenset([x["Môn 1"], x["Môn 2"], x["Môn 3"]]), []),
        return_dtype=pl.List(pl.String)
    )
    .alias("Tổ hợp")
).explode("Tổ hợp")

# Hiển thị kết quả "tinh khiết"
df_final_step_2.head(10).collect()

SOBAODANH,Danh sách môn thi,Tổ hợp 3 môn,Môn 1,Môn 2,Môn 3,Tổ hợp
str,list[str],list[str],str,str,str,str
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Văn"", ""Hóa""]","""Toán""","""Văn""","""Hóa""","""C02"""
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Văn"", ""Sinh""]","""Toán""","""Văn""","""Sinh""","""B03"""
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Hóa"", ""Sinh""]","""Toán""","""Hóa""","""Sinh""","""B00"""
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Văn"", ""Hóa"", ""Sinh""]","""Văn""","""Hóa""","""Sinh""","""C08"""
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01"""
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Hóa""]","""Toán""","""Văn""","""Hóa""","""C02"""
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Lí"", ""Hóa""]","""Toán""","""Lí""","""Hóa""","""A00"""
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Văn"", ""Lí"", ""Hóa""]","""Văn""","""Lí""","""Hóa""","""C05"""
"""01000003""","[""Toán"", ""Văn"", ""Lí""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01"""


In [26]:
df_final_step_2.filter(pl.col("SOBAODANH") == "01000009").head(10).collect()

SOBAODANH,Danh sách môn thi,Tổ hợp 3 môn,Môn 1,Môn 2,Môn 3,Tổ hợp
str,list[str],list[str],str,str,str,str
"""01000009""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01"""
"""01000009""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Hóa""]","""Toán""","""Văn""","""Hóa""","""C02"""
"""01000009""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Lí"", ""Hóa""]","""Toán""","""Lí""","""Hóa""","""A00"""
"""01000009""","[""Toán"", ""Văn"", … ""Hóa""]","[""Văn"", ""Lí"", ""Hóa""]","""Văn""","""Lí""","""Hóa""","""C05"""


## Step 5: Tính toán điểm tổ hợp

In [27]:
df_scores_long = df_score_step_3.unpivot(
    index=["SOBAODANH", "is_eligible"],
    variable_name="Môn",
    value_name="Diem_So"
).filter(pl.col("Diem_So").is_not_null())

df_scores_long.head(10).collect()

SOBAODANH,is_eligible,Môn,Diem_So
str,bool,str,f64
"""01000001""",true,"""Toán""",5.75
"""01000002""",true,"""Toán""",8.0
"""01000003""",false,"""Toán""",6.75
"""01000004""",true,"""Toán""",5.25
"""01000005""",false,"""Toán""",0.0
"""01000006""",true,"""Toán""",6.5
"""01000007""",true,"""Toán""",6.35
"""01000008""",false,"""Toán""",0.0
"""01000009""",true,"""Toán""",7.5


In [28]:
df_final = df_final_step_2
print(df_final.head(12).collect())

shape: (12, 7)
┌───────────┬───────────────────────────┬─────────────────────────┬───────┬───────┬───────┬────────┐
│ SOBAODANH ┆ Danh sách môn thi         ┆ Tổ hợp 3 môn            ┆ Môn 1 ┆ Môn 2 ┆ Môn 3 ┆ Tổ hợp │
│ ---       ┆ ---                       ┆ ---                     ┆ ---   ┆ ---   ┆ ---   ┆ ---    │
│ str       ┆ list[str]                 ┆ list[str]               ┆ str   ┆ str   ┆ str   ┆ str    │
╞═══════════╪═══════════════════════════╪═════════════════════════╪═══════╪═══════╪═══════╪════════╡
│ 01000001  ┆ ["Toán", "Văn", … "Sinh"] ┆ ["Toán", "Văn", "Hóa"]  ┆ Toán  ┆ Văn   ┆ Hóa   ┆ C02    │
│ 01000001  ┆ ["Toán", "Văn", … "Sinh"] ┆ ["Toán", "Văn", "Sinh"] ┆ Toán  ┆ Văn   ┆ Sinh  ┆ B03    │
│ 01000001  ┆ ["Toán", "Văn", … "Sinh"] ┆ ["Toán", "Hóa", "Sinh"] ┆ Toán  ┆ Hóa   ┆ Sinh  ┆ B00    │
│ 01000001  ┆ ["Toán", "Văn", … "Sinh"] ┆ ["Văn", "Hóa", "Sinh"]  ┆ Văn   ┆ Hóa   ┆ Sinh  ┆ C08    │
│ 01000002  ┆ ["Toán", "Văn", … "Hóa"]  ┆ ["Toán", "Văn", "Lí"]   ┆ Toán  ┆ 

In [29]:
# Cách viết chuẩn và an toàn nhất
df_final = df_final_step_2 # Bảng này nên giữ lại is_eligible từ đầu

for i in range(1, 4):
    df_final = df_final.join(
        # Chỉ lấy 3 cột cần thiết để Join, loại bỏ is_eligible ở bảng này để tránh trùng
        df_scores_long.select(["SOBAODANH", "Môn", "Diem_So"]),
        left_on=["SOBAODANH", f"Môn {i}"],
        right_on=["SOBAODANH", "Môn"],
        how="left"
    ).rename({"Diem_So": f"Điểm {i}"})



# Tính tổng điểm trước khi Join
df_final = df_final.with_columns(
    (pl.col("Điểm 1").fill_null(0) +
     pl.col("Điểm 2").fill_null(0) +
     pl.col("Điểm 3").fill_null(0)).round(2).alias("Tổng điểm")
)

# Join Full và xử lý hợp nhất cột SBD ngay lập tức
df_final = (
    df_final.join(
        df_score_step_3.select(["SOBAODANH", "is_eligible"]),
        on="SOBAODANH",
        how="full"
    )
    .with_columns([
        pl.coalesce(["SOBAODANH", "SOBAODANH_right"]).alias("SOBAODANH"),

        pl.col("is_eligible").fill_null(False).alias("Hợp lệ"),

        pl.lit(True).alias("Chương trình mới")
    ])
    .drop(["SOBAODANH_right", "is_eligible"])
)

df_final.head(10).collect()

SOBAODANH,Danh sách môn thi,Tổ hợp 3 môn,Môn 1,Môn 2,Môn 3,Tổ hợp,Điểm 1,Điểm 2,Điểm 3,Tổng điểm,Hợp lệ,Chương trình mới
str,list[str],list[str],str,str,str,str,f64,f64,f64,f64,bool,bool
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Văn"", ""Hóa""]","""Toán""","""Văn""","""Hóa""","""C02""",5.75,7.75,7.75,21.25,true,true
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Văn"", ""Sinh""]","""Toán""","""Văn""","""Sinh""","""B03""",5.75,7.75,8.25,21.75,true,true
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Hóa"", ""Sinh""]","""Toán""","""Hóa""","""Sinh""","""B00""",5.75,7.75,8.25,21.75,true,true
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Văn"", ""Hóa"", ""Sinh""]","""Văn""","""Hóa""","""Sinh""","""C08""",7.75,7.75,8.25,23.75,true,true
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01""",8.0,8.25,8.5,24.75,true,true
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Hóa""]","""Toán""","""Văn""","""Hóa""","""C02""",8.0,8.25,6.75,23.0,true,true
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Lí"", ""Hóa""]","""Toán""","""Lí""","""Hóa""","""A00""",8.0,8.5,6.75,23.25,true,true
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Văn"", ""Lí"", ""Hóa""]","""Văn""","""Lí""","""Hóa""","""C05""",8.25,8.5,6.75,23.5,true,true
"""01000003""","[""Toán"", ""Văn"", ""Lí""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01""",6.75,8.5,8.75,24.0,false,true


In [30]:
print(df_final.columns)

C:\Users\musba\AppData\Local\Temp\ipykernel_7016\1097095893.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(df_final.columns)


['SOBAODANH', 'Danh sách môn thi', 'Tổ hợp 3 môn', 'Môn 1', 'Môn 2', 'Môn 3', 'Tổ hợp', 'Điểm 1', 'Điểm 2', 'Điểm 3', 'Tổng điểm', 'Hợp lệ', 'Chương trình mới']


In [31]:
df_final.filter(
    pl.col("Chương trình mới") == False
).head(5).collect()

SOBAODANH,Danh sách môn thi,Tổ hợp 3 môn,Môn 1,Môn 2,Môn 3,Tổ hợp,Điểm 1,Điểm 2,Điểm 3,Tổng điểm,Hợp lệ,Chương trình mới
str,list[str],list[str],str,str,str,str,f64,f64,f64,f64,bool,bool


In [32]:
sbd_nhieu_to_hop = (
    df_final
    .group_by("SOBAODANH")
    .len() # Đếm số dòng cho mỗi SBD
    .filter(pl.col("len") > 4)
    .sort("len", descending=True)
)

sbd_nhieu_to_hop.head().collect()

SOBAODANH,len
str,u32
"""29024669""",8
"""41012267""",8
"""01046244""",8
"""31005489""",8
"""51016482""",8


In [33]:
# Tính toán thống kê: Số lượng, Trung bình, Độ lệch chuẩn
tohop_summary = (
    df_final
    .group_by("Tổ hợp")
    .agg([
        pl.len().alias("Số lượng"),
        pl.col("Tổng điểm").mean().alias("Trung bình"),
        pl.col("Tổng điểm").std().alias("Độ lệch chuẩn")
    ])
    .sort("Số lượng", descending=True)
)

# Thu thập kết quả và hiển thị
summary_result = tohop_summary.head(10).collect()
print(summary_result)

shape: (10, 4)
┌────────┬──────────┬────────────┬───────────────┐
│ Tổ hợp ┆ Số lượng ┆ Trung bình ┆ Độ lệch chuẩn │
│ ---    ┆ ---      ┆ ---        ┆ ---           │
│ str    ┆ u32      ┆ f64        ┆ f64           │
╞════════╪══════════╪════════════╪═══════════════╡
│ C03    ┆ 481293   ┆ 17.252866  ┆ 3.551566      │
│ C04    ┆ 476472   ┆ 17.263565  ┆ 3.642877      │
│ D01    ┆ 351848   ┆ 18.558829  ┆ 2.940955      │
│ C01    ┆ 347599   ┆ 20.035323  ┆ 3.297001      │
│ C00    ┆ 297403   ┆ 19.719889  ┆ 4.27019       │
│ A07    ┆ 297403   ┆ 16.702326  ┆ 4.018755      │
│ X01    ┆ 246401   ┆ 18.460722  ┆ 2.978974      │
│ C14    ┆ 246401   ┆ 18.460722  ┆ 2.978974      │
│ C02    ┆ 240135   ┆ 19.027154  ┆ 3.572179      │
│ C05    ┆ 162206   ┆ 20.243376  ┆ 3.500692      │
└────────┴──────────┴────────────┴───────────────┘


In [34]:
# Tính toán thống kê: Số lượng, Trung bình, Độ lệch chuẩn
tohop_summary = (
    df_final
    .filter(pl.col("Hợp lệ") == True)
    .group_by("Tổ hợp")
    .agg([
        pl.len().alias("Số lượng"),
        pl.col("Tổng điểm").mean().alias("Trung bình"),
        pl.col("Tổng điểm").std().alias("Độ lệch chuẩn")
    ])
    .sort("Số lượng", descending=True)
)

# Thu thập kết quả và hiển thị
summary_result = tohop_summary.head(10).collect()
print(summary_result)

shape: (10, 4)
┌────────┬──────────┬────────────┬───────────────┐
│ Tổ hợp ┆ Số lượng ┆ Trung bình ┆ Độ lệch chuẩn │
│ ---    ┆ ---      ┆ ---        ┆ ---           │
│ str    ┆ u32      ┆ f64        ┆ f64           │
╞════════╪══════════╪════════════╪═══════════════╡
│ C03    ┆ 474299   ┆ 17.283786  ┆ 3.533936      │
│ C04    ┆ 470149   ┆ 17.2935    ┆ 3.628476      │
│ D01    ┆ 349692   ┆ 18.584643  ┆ 2.912796      │
│ C01    ┆ 341963   ┆ 20.07974   ┆ 3.24138       │
│ C00    ┆ 292489   ┆ 19.697232  ┆ 4.246346      │
│ A07    ┆ 292489   ┆ 16.738354  ┆ 4.012231      │
│ C14    ┆ 244167   ┆ 18.452617  ┆ 2.968719      │
│ X01    ┆ 244167   ┆ 18.452617  ┆ 2.968719      │
│ C02    ┆ 236492   ┆ 19.113284  ┆ 3.49322       │
│ C05    ┆ 160491   ┆ 20.322067  ┆ 3.418802      │
└────────┴──────────┴────────────┴───────────────┘


# Tích hợp với các thí sinh thi theo chương trình cũ

In [35]:
df_old = pd.read_excel("./input/20250715-ketquathi-ct2006.xlsx")

df_old.to_parquet("./output/20250715-ketquathi-ct2006.parquet")

In [36]:
# Quét file Parquet Mus đã có
df_old = (
    pl.scan_parquet("./output/20250715-ketquathi-ct2006.parquet")
    .select(["SOBAODANH"])
    .with_columns([
        # Đảm bảo tuyệt đối SBD là String 8 ký tự
        pl.col("SOBAODANH").cast(pl.String).str.zfill(8),
        # Gán nhãn False cho chương trình cũ
        pl.lit(False).alias("Chương trình mới")
    ])
)

# Join Full với bảng kết quả hiện tại
df_final = (
    df_final.join(df_old, on="SOBAODANH", how="full")
    .with_columns([
        # Gộp SBD để không bị null
        pl.coalesce(["SOBAODANH", "SOBAODANH_right"]).alias("SOBAODANH"),
        # Logic fill: Nếu không có sẵn giá trị (từ bảng cũ) thì mặc định False
        pl.col("Chương trình mới").fill_null(False)
    ])
    .drop("SOBAODANH_right")
)

In [37]:
df_final.head().collect()

SOBAODANH,Danh sách môn thi,Tổ hợp 3 môn,Môn 1,Môn 2,Môn 3,Tổ hợp,Điểm 1,Điểm 2,Điểm 3,Tổng điểm,Hợp lệ,Chương trình mới,Chương trình mới_right
str,list[str],list[str],str,str,str,str,f64,f64,f64,f64,bool,bool,bool
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Văn"", ""Hóa""]","""Toán""","""Văn""","""Hóa""","""C02""",5.75,7.75,7.75,21.25,true,true,null
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Văn"", ""Sinh""]","""Toán""","""Văn""","""Sinh""","""B03""",5.75,7.75,8.25,21.75,true,true,null
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Toán"", ""Hóa"", ""Sinh""]","""Toán""","""Hóa""","""Sinh""","""B00""",5.75,7.75,8.25,21.75,true,true,null
"""01000001""","[""Toán"", ""Văn"", … ""Sinh""]","[""Văn"", ""Hóa"", ""Sinh""]","""Văn""","""Hóa""","""Sinh""","""C08""",7.75,7.75,8.25,23.75,true,true,null
"""01000002""","[""Toán"", ""Văn"", … ""Hóa""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01""",8.0,8.25,8.5,24.75,true,true,null


In [38]:
df_final.filter(
    (pl.col("Hợp lệ") == False) &
    (pl.col("Chương trình mới") == True)
).head(10).collect()

SOBAODANH,Danh sách môn thi,Tổ hợp 3 môn,Môn 1,Môn 2,Môn 3,Tổ hợp,Điểm 1,Điểm 2,Điểm 3,Tổng điểm,Hợp lệ,Chương trình mới,Chương trình mới_right
str,list[str],list[str],str,str,str,str,f64,f64,f64,f64,bool,bool,bool
"""01000003""","[""Toán"", ""Văn"", ""Lí""]","[""Toán"", ""Văn"", ""Lí""]","""Toán""","""Văn""","""Lí""","""C01""",6.75,8.5,8.75,24.0,false,true,null
"""01000005""","[""Toán"", ""Văn"", … ""Địa""]","[""Toán"", ""Văn"", ""Sử""]","""Toán""","""Văn""","""Sử""","""C03""",0.0,7.0,5.5,12.5,false,true,null
"""01000005""","[""Toán"", ""Văn"", … ""Địa""]","[""Toán"", ""Văn"", ""Địa""]","""Toán""","""Văn""","""Địa""","""C04""",0.0,7.0,6.25,13.25,false,true,null
"""01000005""","[""Toán"", ""Văn"", … ""Địa""]","[""Toán"", ""Sử"", ""Địa""]","""Toán""","""Sử""","""Địa""","""A07""",0.0,5.5,6.25,11.75,false,true,null
"""01000005""","[""Toán"", ""Văn"", … ""Địa""]","[""Văn"", ""Sử"", ""Địa""]","""Văn""","""Sử""","""Địa""","""C00""",7.0,5.5,6.25,18.75,false,true,null
"""01000008""","[""Toán"", ""Văn"", … ""Địa""]","[""Toán"", ""Văn"", ""Sử""]","""Toán""","""Văn""","""Sử""","""C03""",0.0,6.25,4.0,10.25,false,true,null
"""01000008""","[""Toán"", ""Văn"", … ""Địa""]","[""Toán"", ""Văn"", ""Địa""]","""Toán""","""Văn""","""Địa""","""C04""",0.0,6.25,4.0,10.25,false,true,null
"""01000008""","[""Toán"", ""Văn"", … ""Địa""]","[""Toán"", ""Sử"", ""Địa""]","""Toán""","""Sử""","""Địa""","""A07""",0.0,4.0,4.0,8.0,false,true,null
"""01000008""","[""Toán"", ""Văn"", … ""Địa""]","[""Văn"", ""Sử"", ""Địa""]","""Văn""","""Sử""","""Địa""","""C00""",6.25,4.0,4.0,14.25,false,true,null


# Lưu file csv, parquet

In [39]:
# Đường dẫn lưu file
parquet_path = "./output/bang_diem_to_hop_2025.parquet"

print("🚀 Đang 'kết tinh' dữ liệu và ghi ra parquet... Đợi xíu nhé!")

# Ghi file với các cột quan trọng nhất
(
    df_final
    .select([
        "SOBAODANH",
        "Tổ hợp",
        "Tổng điểm",
        "Hợp lệ",
        "Chương trình mới"
    ])
    .collect()
    .write_parquet(
        parquet_path,
        compression="zstd",
        compression_level=6,
        use_pyarrow=True
    )
)

print(f"✅ Xong! File đã nằm tại: {parquet_path}")

🚀 Đang 'kết tinh' dữ liệu và ghi ra parquet... Đợi xíu nhé!
✅ Xong! File đã nằm tại: ./output/bang_diem_to_hop_2025.parquet


# Thử truy vấn

In [40]:
df_final.select(pl.len()).collect()

len
u32
5311133


In [41]:
# CHỈ COLLECT 1 LẦN DUY NHẤT VÀ GÁN VÀO BIẾN
df_result = df_final.collect()

# Sau đó in ra từ df_result (lúc này đã nằm trên RAM, tốc độ tức thì)
total = len(df_result)
eligible = df_result.filter(pl.col("Hợp lệ") == True).height
not_eligible = total - eligible

print(f"""
Tổng số bản ghi: {total:,}
Đủ điều kiện: {eligible:,} ({eligible/total:.2%})
Không đủ điều kiện: {not_eligible:,} ({not_eligible/total:.2%})
""")


Tổng số bản ghi: 5,311,133
Đủ điều kiện: 5,236,591 (98.60%)
Không đủ điều kiện: 74,542 (1.40%)



In [42]:
has_combs = df_result.filter(
    pl.col("Tổ hợp").is_not_null()
).height

print(f"Số lượng bản ghi có tổ hợp: {has_combs}")

Số lượng bản ghi có tổ hợp: 5284737


In [43]:
has_combs_eligible = df_result.filter(
    (pl.col("Tổ hợp").is_not_null()) &
    (pl.col("Hợp lệ"))
).height

print(f"Số lượng bản ghi có tổ hợp và các thí sinh tốt nghiệp: {has_combs_eligible}")

Số lượng bản ghi có tổ hợp và các thí sinh tốt nghiệp: 5232848
